# Retraining Cellpose on Custom Data

## Overview

[Website](https://www.cellpose.org) | [GitHub](https://github.com/mouseland/cellpose) | [Paper](https://www.biorxiv.org/content/10.1101/2025.04.28.651001v1) | [Cellpose Documentation](https://cellpose.readthedocs.io/en/latest/index.html) | [Cellpose API](https://cellpose.readthedocs.io/en/latest/api.html#)

In this section, we will walk through how to **retrain Cellpose on your own data**. This is useful when the default models don’t perform well on your specific cell type, staining method, or imaging modality.

Retraining allows Cellpose to learn directly from your examples, leading to better segmentation accuracy and more relevant masks for your experiments.

To go through the training process, you need **pairs of raw microscopy images and their corresponding label masks**. The raw images are what you want to segment, and the label masks are the ground truth segmentations that the model will learn from.

## Make sure you have GPU access

To Enable GPU:

1 - Navigate to `Runtime -> Change Runtime Type`

2 - Select `Python 3` as `Runtime Type`

3 - Select one available GPU (e.g. `T4 GPU`) as `Hardware accelerator`.

<br>

<div align="left"> <img src="https://raw.githubusercontent.com/bobiac/bobiac-book/main/_static/images/cellpose/colab_runtime.png" alt="Ilastik Logo" width="400"></div>

## Mount your google drive

To access the data for the course you first need to mount your Google Drive.

Run the cell below to connect your Google Drive to colab and follow the instructions to authenticate your Google account.

You will need to allow access to your Google Drive so that the notebook can read and write files.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Then click on `folder icon` on the left bar, press the `refresh button`. Your Google Drive folder should now be available here (e.g. MyDrive).

<div align="left"> <img src="https://raw.githubusercontent.com/bobiac/bobiac-book/main/_static/images/cellpose/colab_folder.png" alt="Ilastik Logo" width="300"></div>

## Download the Data

For this tutorial, we will use the sample dataset provided by Cellpose, which includes both training and test images pairs.  

Run the cell below to download the data for this exercise and save it in you Google Drive. A new folder called `bobiac_data_cellpose` will be created in your Google Drive.

<p class="alert alert-info">
    <strong>🚧 Note:</strong> We will use only the sample dataset provided by Cellpose to demonstrate the retraining pipeline. If you want to retrain Cellpose on your own data, you will probably need more image pairs for training and for testing, as well as some validation data.
</p>

In [ ]:
# Create directory
!mkdir -p /content/bobiac_data_cellpose
# Download the data
!wget https://github.com/bobiac/bobiac-book/releases/download/data-bobiac-2026/04_segmentation_cellpose_training.zip -O /content/bobiac_data_cellpose/04_segmentation_cellpose_training.zip
# Unzip the data, remove zip file and macOS metadata files (if any)
!cd /content/bobiac_data_cellpose && unzip 04_segmentation_cellpose_training.zip && rm -f 04_segmentation_cellpose_training.zip && rm -rf __MACOSX

## Install Cellpose & other dependencies

In [ ]:
!pip install cellpose
!pip install matplotlib

## Creating Label Masks

As mentioned above, the dataset we will use already includes label masks, but if you want to retrain Cellpose on your own data, you will need to create these masks yourself.

There are different tools available. One example is the Cellpose GUI itself, which lets you modify and save updated labels in a user-friendly way. You can see the [Cellpose documentation](https://cellpose.readthedocs.io/en/latest/gui.html#training-your-own-cellpose-model) for more details on how to create and edit label masks.

Another option is to use annotation tools like [napari](https://napari.org/). If using `uv`, you can simply start `napari` by running in your terminal:

`uvx "napari[all]"`

The `napari` GUI will open and you can load your raw images and create or modify and then save the corresponding label masks using the annotation tools available.

## Retraining Pipeline

The retraining pipeline consists of the following steps:
1. **Data Preparation**: Organize your raw images and label masks into a format that Cellpose can use for training.

2. **Model Configuration**: Set up the training parameters, such as the number of epochs, learning rate, and batch size.

3. **Run the standard model on test data [optional]**: Before training, in this tutorial we will run the standard Cellpose model on the test data to see how it performs before retraining.

4. **Train the Model**: Use the prepared data to train a new Cellpose model.

5. **Evaluate the Model**: After training, evaluate the performance of the new model on the test data and compare it to the performance of the standard model.

<br>

<details>
<summary><b>Retraining for 3D Data</b></summary>
<br>

`train.train_seg` always trains a **2D** network, even for 3D segmentation. When you run inference with `do_3D=True`, Cellpose applies this 2D network to the **XY, XZ and YZ** slices of the volume and combines the resulting flows (see the [3D segmentation documentation](https://cellpose.readthedocs.io/en/latest/do3d.html)).

This means that if your data is 3D and the default/pretrained model doesn't perform well in `do_3D` mode, your training set should **not** consist only of XY slices. To get a model that segments well in all three orientations, you should provide labelled 2D image/mask pairs sampled from all three orthogonal planes of your annotated volumes:

- **XY** slices (the standard top-down view)
- **XZ** slices (a vertical "side" view)
- **YZ** slices (the other vertical "side" view)

Mixing slices from the three planes into your `train` (and `test`) folders teaches the network what objects look like from every orientation, so that the flows it predicts on XZ/YZ slices during 3D inference are as accurate as the ones it predicts on XY slices.

**Generating multi-plane training crops**

Cellpose provides a helper script, [`make_train.py`](https://github.com/MouseLand/cellpose/blob/main/cellpose/gui/make_train.py), that extracts random XY, XZ and YZ crops from a 3D image (and its label volume, if available) and saves them as 2D `.tif` files ready to be labelled (and/or corrected). Note that this script needs a local display, so it can't be run on Colab; run it locally instead.

```bash
python -m cellpose.gui.make_train --dir /path/to/3d/data --anisotropy 5
```

Use `--anisotropy` (the ratio of the Z pixel size to the XY pixel size, the same value you would pass to `model.eval(..., anisotropy=...)`) so that the XZ/YZ crops are rescaled to look isotropic, matching the appearance of the XY crops.

Once you have a folder of labelled 2D image/mask pairs covering all three planes, you can use them with `train.train_seg` exactly as shown below: no extra 3D-specific parameters are needed, since the network itself only ever sees 2D slices.
</details>

## Import Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from cellpose import core, io, metrics, models, train

## Setup

In [ ]:
io.logger_setup()  # to get printing of progress

use_gpu = core.use_gpu()
print("GPU available:", use_gpu)

## Data Handling

Cellpose expects images and their corresponding masks to live **in the same folder**. The file names must share the same prefix and only differ by the suffix. For example, if the raw image is named `img_0.tif`, the corresponding mask should be named `img_0_seg.tif` (`_seg` here is just an example, you can use any suffix you like).

You'll also need to **split your data** into a `train` and a `test` folder. The model learns from the training set and is evaluated on the test set (images it has never seen during training).

```
cellpose_data/
├── train/
│   ├── img_0.tif
│   ├── img_0_seg.tif
│   ├── img_1.tif
│   ├── img_1_seg.tif
│   └── ...
└── test/
    ├── img_8.tif
    ├── img_8_seg.tif
    ├── img_9.tif
    ├── img_9_seg.tif
    └── ...
```

After organizing your data, the first step is to define the `train` and `test` directories, then load the data with [`io.load_train_test_data`](https://cellpose.readthedocs.io/en/latest/api.html#cellpose.io.load_train_test_data), passing the mask (and image if needed) suffixes so Cellpose can pair them correctly.

In [ ]:
ROOT_FOLDER_PATH = Path("bobiac_data_cellpose/04_segmentation_cellpose_training")

train_dir = ROOT_FOLDER_PATH / "train"
test_dir = ROOT_FOLDER_PATH / "test"

# add name filters to select only images and masks from the folders
# `mask_filter` identifies mask files by their suffix
# (e.g. "_seg" for files like "img_000_seg". If not .tif, add also the extension).
mask_filter = "_seg"

# if necessary, you can also specify an `image_filter` to select images with a specific
# suffix (e.g. "_img" for files like "img_000_raw.tif". If not .tif, add also the extension).
# image_filter = "_raw"

# Load training and test data
output = io.load_train_test_data(
    str(train_dir),
    str(test_dir),
    mask_filter=mask_filter,
    # image_filter=image_filter
)

# assign the output to the appropriate variables
train_data, train_labels, _, test_data, test_labels, _ = output

## Initialize the Model

To initialize Cellpose model we can use the `models.CellposeModel()` class.

There are other parameters we can set when initializing the model, here we will only use `pretrained_model` to specify which pretrained model to use and `gpu` to specify whether to use GPU (if available) for faster inference.

Currently, the available pretrained models are:
- `cpsam`: this is the original CellposeSAM model released in April 2025 using the SAM-ViTL backbone (default model)
- `cpsam_v2`: this is the CellposeSAM model released in June 2026 using the SAM-ViTL backbone, it includes a fix in the training for low contrast regions
- `cpdino`: this is the CellposeDINO model released in June 2026 using the DINOv3-ViTL backbone
- `cpdino-vitb`: this is the CellposeDINO model released in June 2026 using the DINOv3-ViTB backbone (smaller model)

<p class="alert alert alert-info">
    <strong>Note:</strong> Only the default <code>cpsam</code> model is downloaded automatically the first time you run this notebook (this may take a while). The other models (<code>cpsam_v2</code>, <code>cpdino</code>, <code>cpdino-vitb</code>) must be downloaded manually, see below.
</p>

### Downloading the Other Pretrained Models

These models are hosted on the [Cellpose-SAM Hugging Face repository](https://huggingface.co/mouseland/cellpose-sam). We need to download the model weights and place them in the `~/.cellpose/models` directory (the `MODEL_DIR` variable from the `cellpose.models` module).

In [ ]:
from cellpose.models import MODEL_DIR
from cellpose.utils import download_url_to_file

model_name = "cpsam_v2"  # or "cpdino" / "cpdino-vitb"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODEL_DIR / model_name
if not model_path.exists():
    url = f"https://huggingface.co/mouseland/cellpose-sam/resolve/main/{model_name}"
    download_url_to_file(url, str(model_path))

In [ ]:
model_path = str(MODEL_DIR / "cpsam_v2")  # or "cpdino" / "cpdino-vitb" or "cpsam"
model = models.CellposeModel(pretrained_model=model_path, gpu=use_gpu)


## How does a model perform on the test data?

Before training a new model, let's see how the pre-trained Cellpose model we chose performs on the test data.

In [ ]:
# run model on test images
masks, _, _ = model.eval(test_data, batch_size=8)

Now we'll quantify how well the standard model segments the test images by comparing the predicted masks to the ground truth labels using [`metrics.average_precision`](https://cellpose.readthedocs.io/en/latest/api.html#cellpose.metrics.average_precision).

For each test image, every predicted mask is matched to a ground truth mask based on their [Intersection over Union (IoU)](https://en.wikipedia.org/wiki/Jaccard_index): the overlap area divided by the union area of the two masks.

<div align="center"> <img src="https://raw.githubusercontent.com/bobiac/bobiac-book/main/_static/images/cellpose/iou.png" alt="iou" width="300"></div>

A predicted mask counts as a **true positive** (TP) if its IoU with a ground truth mask is above a given threshold; otherwise it is a **false positive** (FP), and any unmatched ground truth mask is a **false negative** (FN). The average precision (AP) is then computed as:

$$
AP = \frac{TP}{TP + FP + FN}
$$

By default, `metrics.average_precision` computes this at IoU thresholds of **0.5**, **0.75**, and **0.9**. A higher threshold requires a tighter overlap between predicted and ground truth masks, so AP typically decreases as the threshold increases. Comparing these values before and after retraining gives us a quantitative measure of how much the model improves on our data.

In [ ]:
# check performance using ground truth labels
# average_precision returns AP at IoU thresholds [0.5, 0.75, 0.9] by default
values = metrics.average_precision(test_labels, masks)
average_precision, _, _, _ = values

print(f"average precision at iou threshold 0.5  = {average_precision[:, 0].mean():.3f}")
print(f"average precision at iou threshold 0.75 = {average_precision[:, 1].mean():.3f}")
print(f"average precision at iou threshold 0.9  = {average_precision[:, 2].mean():.3f}")

Let's now visualize the results for the test images.

In [ ]:
n = 0  # test image index to visualize
cyto_ch = 1  # channel index for cytoplasm (0=nucleus, 1=cytoplasm in this dataset)
raw_data = test_data[n][cyto_ch]  # selecting which test data ans which channel
pred_mask = masks[n]  # selecting the predicted mask for the same test image
gt_mask = test_labels[n]  # selecting the ground truth mask for the same test image

plt.figure(figsize=(10, 5))

plt.subplot(1, 3, 1)
plt.imshow(raw_data, cmap="gray")
plt.title(f"Test Image {n}")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(pred_mask, cmap="nipy_spectral")
plt.title(f"Predicted Mask {n}")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(gt_mask, cmap="nipy_spectral")
plt.title(f"GT Mask {n}")
plt.axis("off")

plt.tight_layout()
plt.show()

## Train New Model

Now we're ready to retrain Cellpose using the `train_seg` method from the `train` module.

Here we will change few training parameters but you can find the full parameters description for the `train.train_seg` method in the dropdown below or in the [Cellpose API documentation](https://cellpose.readthedocs.io/en/latest/api.html#module-cellpose.train)

<details>
<summary><b>Cellpose <code>train.train_seg()</code> Parameters</b></summary>
<br>

**Input Data**

| Parameter | Type | Default | What it does |
|---|---|---|---|
| `train_data` | `list[np.ndarray] \| None` | `None` | List of numpy arrays (2D or 3D images). Mutually exclusive with `train_files`, use one or the other. |
| `train_labels` | `list[np.ndarray] \| None` | `None` | List of integer label arrays matching `train_data`. `0` = bzackground, `1, 2, ...` = individual masks. |
| `train_files` | `list[str] \| None` | `None` | File paths to training images. Used instead of `train_data` when loading from disk. Use together with `load_files=True` (the default), this is the standard path when working from disk. |
| `train_labels_files` | `list[str] \| None` | `None` | File paths to the corresponding label files. |
| `test_data` / `test_labels` | `list[np.ndarray] \| None` | `None` | Same as above but for validation. Used only to report test loss, does not affect gradient updates. |
| `test_files` / `test_labels_files` | `list[str] \| None` | `None` | Same as above, file-based version. |
| `load_files` | `bool` | `True` | If `True`, loads images/labels from the `*_files` paths at the start of training. Set to `False` if you've already loaded them into arrays and don't want redundant I/O. |
| `channel_axis` | `int \| None` | `None` | Which axis in the arrays is the channel axis. `None` = infer automatically. |

**Sampling & Epoch Control**

| Parameter | Type | Default | What it does |
|---|---|---|---|
| `train_probs` | `np.ndarray \| None` | `None` | Per-image sampling probability for each epoch. If `None`, all images get equal weight (`1/n`). The right lever if you want to hard-mine difficult images or balance an uneven dataset. Must sum to 1 after normalization. |
| `test_probs` | `np.ndarray \| None` | `None` | Same for test images. |
| `nimg_per_epoch` | `int \| None` | `None` | How many images (with random augmentation each time) Cellpose samples per epoch, doing one gradient update per image. If `None`, defaults to `len(train_data)`. For a large dataset (50/100+ images), leave as `None`, one pass per epoch is already enough gradient steps. For a small dataset (5-10 images), set it higher than your dataset size (e.g. `50` or `100` for 5 images): each image gets sampled multiple times per epoch with different random augmentations (crop, rotation, flip, scale), and more updates/epoch help the linear LR warmup over the first 10 epochs actually move the weights.
| `nimg_test_per_epoch` | `int \| None` | `None` | Same as `nimg_per_epoch` but for test loss reporting only, it does not affect training. For a small test set (2-5 images), leave as `None` (uses all) for a stable loss estimate, subsampling a tiny test set gives noisy numbers. For a large test set (100+ images), set it lower (e.g. `20`) to speed up per-epoch evaluation at the cost of noisier test loss curves. |
| `min_train_masks` | `int` | `5` | Images with fewer than this many mask instances are silently dropped before training starts, guarding against nearly-empty label images corrupting training. If your dataset is small, check how many images actually survive this filter. |

**Optimizer**

| Parameter | Type | Default | What it does |
|---|---|---|---|
| `learning_rate` | `float` | `1e-5` | learning rate (LR) for the AdamW algorithm used by Cellpose. Controls how large each weight update is. Too high → the model overshoots and training becomes unstable (loss oscillates). Too low → the model learns very slowly or gets stuck. For fine-tuning a pre-trained model like Cellpose, a small value is preferred to avoid erasing what the model already knows (e.g. `1e-5` or 0.00001). Note that `learning_rate` is also ramped up linearly from 0 to its target value over the first 10 epochs (regardless of `n_epochs`), so with very few epochs the target learning rate may never be reached. |
| `weight_decay` | `float` | `0.1` | AdamW algorithm weight decay (L2 regularization). `0.1` is the modern default from the AdamW paper. |
| `SGD` | `bool` | `False` | Deprecated as of v4.0.1. AdamW is always used regardless of this value. |
| `n_epochs` | `int` | `100` | one epoch = one full pass through all training images. More epochs give the model more time to learn, but too many can lead to *overfitting*: the model memorizes the training data and performs worse on new images. Watch the test loss, if it starts rising while the train loss keeps falling, you've trained too long. |
| `batch_size` | `int` | `1` | number of `bsize`x`bsize` pixels image tiles processed simultaneously on the GPU before the model updates its weights (the image is split into tiles before being processed). Larger batches give more stable gradient estimates but require more memory. Smaller batches are noisier but work well for small datasets and use less GPU memory. See the [cellpose_notebook](.//cellpose_notebook.ipynb) for more details on batch size and GPU performance.|
| `bsize` | `int` | `256` | Size of each tile in pixels (`bsize × bsize`). For the SAM-based models (`cpsam`, `cpsam_v2`) this is fixed to 256x256, don't override it. For the DINO-based models (`cpdino`, `cpdino-vitb`) the default is 384 and can be adjusted. |

**Augmentation & Preprocessing**

| Parameter | Type | Default | What it does |
|---|---|---|---|
| `normalize` | `bool \| dict` | `True` | If `True`, applies default percentile normalization per image. If a dict, merges with default normalize params (e.g. `{"tile_norm_blocksize": 128}` for tile-based normalization on large images). |
| `compute_flows` | `bool` | `False` | If `True`, recomputes flow fields from label masks during training rather than using cached flows. Slower but ensures flows match labels exactly if labels were recently edited. |
| `rescale` | `bool` | `False` | If `True`, rescales each image during training based on its object diameter relative to the model's expected diameter (`net.diam_mean`). Helps when objects in your data differ significantly in size from the training distribution. |
| `scale_range` | `float \| None` | `None` | Range of random scale augmentation passed to `random_rotate_and_resize()`. Default resolves to `0.5`, meaning scale is sampled in `[1-0.5, 1+0.5]`. Larger values = more aggressive scale augmentation. |

**Loss**

| Parameter | Type | Default | What it does |
|---|---|---|---|
| `class_weights` | `np.ndarray \| None` | `None` | Array of per-class weights passed to `nn.CrossEntropyLoss(weight=...)`. Use this to up-weight underrepresented mask classes. Converted to a float32 CUDA tensor internally. |

**Saving**

| Parameter | Type | Default | What it does |
|---|---|---|---|
| `save_path` | `str \| Path \| None` | `None` | Directory where the trained model is written. If `None`, no model is saved to disk. |
| `save_every` | `int` | `100` | Save a checkpoint every N epochs. |
| `save_each` | `bool` | `False` | If `True`, each checkpoint gets a unique filename (epoch-stamped). If `False`, each save overwrites the previous checkpoint. |
| `model_name` | `str \| None` | `None` | The filename stem for the saved model. If `None`, an automatic name is generated. |

</details>

In [ ]:
# path and name for saving the trained model
save_path = ROOT_FOLDER_PATH
model_name = "new_model"

# Training params - here we only change the number of epochs and images per epoch
# but you can change other parameters as well, see the dropdown above or the Cellpose\
# API documentation for details.

n_epochs = 10  # using 10 to speed up the training for this tutorial
nimg_per_epoch = 5  # using 5 to speed up the training for this tutorial

new_model_path, train_losses, test_losses = train.train_seg(
    model.net,
    train_data=train_data,
    train_labels=train_labels,
    test_data=test_data,
    test_labels=test_labels,
    n_epochs=n_epochs,
    nimg_per_epoch=nimg_per_epoch,
    model_name=model_name,
    save_path=save_path,
    load_files=False,  # we already loaded the data above with `io.load_train_test_data`
)

# NOTE: to speed up the training you can omit the test data and test labels from the
# `train_seg` function, but then you won't get test losses or a model saved at the epoch
# with the best test loss.

We can also plot the training and test losses over epochs to see how the model is learning.

In [ ]:
plt.plot(train_losses, label="train loss")
plt.plot(test_losses, label="test loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Test Losses")
plt.legend()
plt.show()

## Evaluate on test data

To evaluate the new model, we can run it on the test images and compute the average precision again to see (if and) how much it improved compared to the standard model.

In [ ]:
# load the newly trained model
model = models.CellposeModel(pretrained_model=new_model_path, gpu=use_gpu)

# run model on test images
masks, _, _ = model.eval(test_data, batch_size=8)

# check performance using ground truth labels
# average_precision returns AP at IoU thresholds [0.5, 0.75, 0.9] by default
values = metrics.average_precision(test_labels, masks)
average_precision, _, _, _ = values

print(f"average precision at iou threshold 0.5  = {average_precision[:, 0].mean():.3f}")
print(f"average precision at iou threshold 0.75 = {average_precision[:, 1].mean():.3f}")
print(f"average precision at iou threshold 0.9  = {average_precision[:, 2].mean():.3f}")